# 05 — Modeling and Evaluation

This notebook builds baseline machine learning pipelines for customer churn prediction.

Main goals:
- prepare train and test sets
- build reproducible preprocessing pipelines
- compare multiple classification models
- evaluate model performance with suitable classification metrics

In [18]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from pathlib import Path
import json

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)



## 1. Load the dataset and configuration

The raw dataset is loaded again and the saved preprocessing decisions are reused to keep the modeling stage consistent with the previous notebook.

In [19]:
DATA_PATH = Path("../data/raw/telco.csv")

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

Shape: (7043, 50)


,Customer ID,Gender,Age,Under 30,Senior Citizen,Married,Dependents,Number of Dependents,Country,State,...,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Satisfaction Score,Customer Status,Churn Label,Churn Score,CLTV,Churn Category,Churn Reason
0,8779-QRDMV,Male,78,No,Yes,No,No,0,United States,California,...,20,0.00,59.65,3,Churned,Yes,91,5433,Competitor,Competitor offered more data
1,7495-OOKFY,Female,74,No,Yes,Yes,Yes,1,United States,California,...,0,390.80,1024.10,3,Churned,Yes,69,5302,Competitor,Competitor made better offer
2,1658-BYGOY,Male,71,No,Yes,No,Yes,3,United States,California,...,0,203.94,1910.88,2,Churned,Yes,81,3179,Competitor,Competitor made better offer
3,4598-XLKNJ,Female,78,No,Yes,Yes,Yes,1,United States,California,...,0,494.00,2995.07,2,Churned,Yes,88,5337,Dissatisfaction,Limited range of services
4,4846-WHAFZ,Female,80,No,Yes,Yes,Yes,1,United States,California,...,0,234.21,3102.36,2,Churned,Yes,67,2793,Price,Extra data charges


In [20]:
CONFIG_PATH = Path("../reports/feature_config.json")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    feature_config = json.load(f)

feature_config

{'target_column': 'target_churn',
 'dropped_columns': ['Customer Status',
  'Churn Category',
  'Churn Reason',
  'Churn Score',
  'Customer ID',
  'Country',
  'State',
  'City',
  'Zip Code',
  'Latitude',
  'Longitude',
  'Quarter',
  'CLTV',
  'Total Revenue',
  'Churn Label'],
 'numeric_features': ['Age',
  'Number of Dependents',
  'Population',
  'Number of Referrals',
  'Tenure in Months',
  'Avg Monthly Long Distance Charges',
  'Avg Monthly GB Download',
  'Monthly Charge',
  'Total Charges',
  'Total Refunds',
  'Total Extra Data Charges',
  'Total Long Distance Charges',
  'Satisfaction Score'],
 'categorical_features': ['Gender',
  'Under 30',
  'Senior Citizen',
  'Married',
  'Dependents',
  'Referred a Friend',
  'Offer',
  'Phone Service',
  'Multiple Lines',
  'Internet Service',
  'Internet Type',
  'Online Security',
  'Online Backup',
  'Device Protection Plan',
  'Premium Tech Support',
  'Streaming TV',
  'Streaming Movies',
  'Streaming Music',
  'Unlimited Data

## 2. Recreate the baseline modeling dataset

The target variable and the preprocessing decisions from the previous notebook are applied again here.

The saved configuration helps keep column exclusions and feature groups consistent, while the raw dataset is prepared again to keep this notebook reproducible and self-contained.

In [21]:
df["target_churn"] = df["Churn Label"].map({"No": 0, "Yes": 1})

object_cols = df.select_dtypes(include="object").columns.tolist()
df[object_cols] = df[object_cols].replace("None", np.nan)
df[object_cols] = df[object_cols].replace(r"^\s*$", np.nan, regex=True)

df[["Churn Label", "target_churn"]].head()

,Churn Label,target_churn
0,Yes,1
1,Yes,1
2,Yes,1
3,Yes,1
4,Yes,1


In [22]:
drop_cols = feature_config["dropped_columns"]

baseline_df = df.drop(columns=drop_cols).copy()

print("Baseline shape:", baseline_df.shape)
baseline_df.head()

Baseline shape: (7043, 36)


,Gender,Age,Under 30,Senior Citizen,Married,Dependents,Number of Dependents,Population,Referred a Friend,Number of Referrals,...,Contract,Paperless Billing,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Satisfaction Score,target_churn
0,Male,78,No,Yes,No,No,0,68701,No,0,...,Month-to-Month,Yes,Bank Withdrawal,39.65,39.65,0.00,20,0.00,3,1
1,Female,74,No,Yes,Yes,Yes,1,55668,Yes,1,...,Month-to-Month,Yes,Credit Card,80.65,633.30,0.00,0,390.80,3,1
2,Male,71,No,Yes,No,Yes,3,47534,No,0,...,Month-to-Month,Yes,Bank Withdrawal,95.45,1752.55,45.61,0,203.94,2,1
3,Female,78,No,Yes,Yes,Yes,1,27778,Yes,1,...,Month-to-Month,Yes,Bank Withdrawal,98.50,2514.50,13.43,0,494.00,2,1
4,Female,80,No,Yes,Yes,Yes,1,26265,Yes,1,...,Month-to-Month,Yes,Bank Withdrawal,76.50,2868.15,0.00,0,234.21,2,1


## 2.1 Define features and target

The baseline dataset is split into input features (`X`) and target labels (`y`) for modeling.

In [23]:
X = baseline_df.drop(columns=["target_churn"])
y = baseline_df["target_churn"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 35)
y shape: (7043,)


## 2.2 Identify feature groups

Numeric and categorical feature groups are loaded from the saved preprocessing configuration.

These groups will be used to apply different preprocessing steps inside the modeling pipeline.

In [24]:
numeric_features = feature_config["numeric_features"]
categorical_features = feature_config["categorical_features"]

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nNumeric columns:")
print(numeric_features)

print("\nCategorical columns:")
print(categorical_features)

Numeric features: 13
Categorical features: 22

Numeric columns:
['Age', 'Number of Dependents', 'Population', 'Number of Referrals', 'Tenure in Months', 'Avg Monthly Long Distance Charges', 'Avg Monthly GB Download', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Satisfaction Score']

Categorical columns:
['Gender', 'Under 30', 'Senior Citizen', 'Married', 'Dependents', 'Referred a Friend', 'Offer', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Internet Type', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method']


## 3. Train-test split

The dataset is divided into training and test sets.

The test set is reserved for final evaluation, while model comparison is supported with cross-validation on the training set.

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5634, 35)
X_test : (1409, 35)
y_train: (5634,)
y_test : (1409,)


## 4. Cross-validation setup

To make the model comparison more reliable, cross-validation is applied on the training set.

This allows the baseline models to be compared more robustly before the final test-set evaluation.

In [26]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

### Why cross-validation was used instead of a separate validation split

A separate validation set could have been created, but this project uses cross-validation on the training data instead.

Because the dataset is moderate in size, this approach allows more efficient use of the available training observations while still providing a more stable comparison across models.

The test set is kept untouched and reserved only for final evaluation.

## 5. Build preprocessing pipelines

Separate preprocessing steps are defined for numeric and categorical features.

These are combined using a `ColumnTransformer` to create a clean and reproducible workflow.

In [27]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

numeric_pipeline

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

In [28]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

categorical_pipeline

Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore'))])

In [29]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['Age', 'Number of Dependents', 'Population',
                                  'Number of Referrals', 'Tenure in Months',
                                  'Avg Monthly Long Distance Charges',
                                  'Avg Monthly GB Download', 'Monthly Charge',
                                  'Total Charges', 'Total Refunds',
                                  'Total Extra Data Charges',
                                  'Total Long Distanc...
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['Gender', 'Under 30', 'Senior Citizen',
                                  'Married', 'Dependents', 'Referred a Friend',
                                  'Offer', 'Phone Service', 'Multiple Lines',
                                  'Internet Service', 'Internet Type',
                                  'Online Security', 'Online Backup',
                                  'Device Protection Plan',
                                  'Premium Tech Support', 'Streaming TV',
                                  'Streaming Movies', 'Streaming Music',
                                  'Unlimited Data', 'Contract',
                                  'Paperless Billing', 'Payment Method'])])

## Numeric Features
* **Median Imputation:** Chosen for robustness against outliers compared to mean imputation.
* **StandardScaler:** Applied to normalize feature magnitudes, ensuring a stable baseline for linear models.

## Categorical Features
* **Most Frequent (Mode) Imputation:** A simple baseline strategy for missing data.
* **One-Hot Encoding:** Used to convert categories to numeric vectors without introducing artificial ordinality.

## Decision Matrix

| Stage | Method | Rationale |
| :--- | :--- | :--- |
| **Num Impute** | Median | Robustness |
| **Num Scale** | StandardScaler | Feature parity |
| **Cat Impute** | Most Frequent | Simplicity |
| **Cat Encode** | One-Hot | No artificial order |


## 6. Define baseline models

The following baseline models are compared:

- **Logistic Regression:** Serves as a linear baseline, highly interpretable.
- **Decision Tree:** Captures non-linear patterns with high transparency.
- **Random Forest:** An ensemble method (Bagging) that reduces variance and improves stability.
- **XGBoost:** A gradient boosting method (Boosting) that iteratively optimizes performance, often yielding the highest accuracy for tabular data.

These models represent different modeling families, ensuring that our baseline comparison is comprehensive and robust.

In [30]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric="logloss")
}

models

{'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
 'Decision Tree': DecisionTreeClassifier(random_state=42),
 'Random Forest': RandomForestClassifier(random_state=42),
 'XGBoost': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric='logloss',
               feature_types=None, feature_weights=None, gamma=None,
               grow_policy=None, importance_type=None,
               interaction_constraints=None, learning_rate=None, max_bin=None,
               max_cat_threshold=None, max_cat_to_onehot=None,
               max_delta_step=None, max_depth=None, max_leaves=None,
               min_child_weight=None, missing=nan, monotone_constraints=None,
               multi_strategy=None, n_estimators=None, n_jobs=None,
               num_parallel_tree=None, ...)}

## 7. Create an evaluation helper function

A reusable function `evaluate_model` is implemented to streamline the evaluation process. This ensures that every model follows the exact same preprocessing and validation protocol.

**Key Features:**
- **Automated Pipeline:** Combines preprocessing and the estimator to prevent data leakage.
- **Cross-Validation:** Uses F1-score as the primary metric during CV to account for potential class imbalance.
- **Comprehensive Metrics:** Calculates Accuracy, Precision, Recall, F1-score, and ROC-AUC for a 360-degree performance view.
- **Robustness:** Includes checks for probability prediction availability to ensure compatibility across different model types.

In [31]:
def evaluate_model(model_name, model, preprocessor, X_train, X_test, y_train, y_test, cv):
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])
    
    cv_scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="f1"
    )
    
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    
    if hasattr(pipeline, "predict_proba"):
        y_proba = pipeline.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    else:
        y_proba = None
        roc_auc = np.nan
    
    results = {
        "model": model_name,
        "cv_f1_mean": cv_scores.mean(),
        "cv_f1_std": cv_scores.std(),
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred),
        "test_recall": recall_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
        "test_roc_auc": roc_auc
    }
    
    return pipeline, y_pred, y_proba, results

## 8. Train and evaluate all baseline models

Each model is trained through the same preprocessing pipeline.

Cross-validated training performance and final test-set metrics are collected for comparison.

In [32]:
all_results = []
trained_pipelines = {}

for model_name, model in models.items():
    pipeline, y_pred, y_proba, results = evaluate_model(
        model_name=model_name,
        model=model,
        preprocessor=preprocessor,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        cv=cv
    )
    
    trained_pipelines[model_name] = {
        "pipeline": pipeline,
        "y_pred": y_pred,
        "y_proba": y_proba
    }
    
    all_results.append(results)

## 9. Compare model performance

The baseline models are compared using cross-validated F1-score and final test-set metrics.

This helps identify which model provides the strongest starting point for further tuning and interpretation.

In [33]:
results_df = pd.DataFrame(all_results).sort_values(by="cv_f1_mean", ascending=False)
results_df.round(4)

,model,cv_f1_mean,cv_f1_std,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
2,Random Forest,0.9310,0.0098,0.9610,0.9761,0.8743,0.9224,0.9840
0,Logistic Regression,0.9264,0.0063,0.9624,0.9547,0.9011,0.9271,0.9922
3,XGBoost,0.9261,0.0031,0.9610,0.9493,0.9011,0.9246,0.9908
1,Decision Tree,0.9015,0.0099,0.9489,0.9081,0.8984,0.9032,0.9328


In [38]:
best_cv_model_name = results_df.iloc[0]["model"]
best_cv_model_name

'Random Forest'

In [42]:
# Selected as the preferred baseline model after comparing cross-validation
# performance, held-out test metrics, and interpretability.
preferred_model_name = "Logistic Regression"
preferred_model_name

'Logistic Regression'

## Baseline model selection note

Random Forest achieved the highest cross-validated F1-score and therefore ranked first in the baseline comparison table.

However, Logistic Regression showed the strongest overall balance on the held-out test set, including the highest test F1-score and ROC-AUC, while also remaining easier to interpret.

For this reason, Random Forest is treated as the best cross-validated baseline, while Logistic Regression is selected as the preferred baseline model for the next stage of interpretation and improvement.

In [40]:
selected_y_pred = trained_pipelines[preferred_model_name]["y_pred"]

print(f"Preferred model: {preferred_model_name}\n")
print(classification_report(y_test, selected_y_pred))

Preferred model: Logistic Regression

              precision    recall  f1-score   support

           0       0.96      0.98      0.97      1035
           1       0.95      0.90      0.93       374

    accuracy                           0.96      1409
   macro avg       0.96      0.94      0.95      1409
weighted avg       0.96      0.96      0.96      1409



## Interpretation of the preferred baseline model

The preferred baseline model, Logistic Regression, showed a strong and well-balanced classification performance on the test set.

The confusion matrix indicates that the model correctly identified most non-churned customers (1019) and most churned customers (337), while keeping both false positives (16) and false negatives (37) relatively low.

For the churn class, the model achieved:
- **Precision = 0.95**
- **Recall = 0.90**
- **F1-score = 0.93**

This suggests that the model is effective at identifying at-risk customers while maintaining a low rate of unnecessary false alarms. As a result, Logistic Regression provides a strong and interpretable baseline for the next stage of the project.

In [41]:
selected_cm = confusion_matrix(y_test, selected_y_pred)
selected_cm

array([[1019,   16],
       [  37,  337]], dtype=int64)

## Conclusion

The baseline comparison showed that Random Forest achieved the highest cross-validated F1-score, while Logistic Regression provided the strongest overall balance on the held-out test set and remained easier to interpret.

At this stage, Logistic Regression is retained as the preferred baseline model, while Random Forest and XGBoost remain strong candidates for the next tuning stage.